# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/martindiarua/ML_01/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

The target is `refresh_priority`, a binary proxy label created in ML-05 from two existing signals: `CTR` and `engagement_rate`. Because those two variables directly define the target, I will exclude them from the model features. I will also exclude the previously identified `leak_feature` and identifier fields such as `content_id` and `client_id`.

The purpose is therefore not to reproduce the rule used to create the proxy label. Instead, the model will test whether other available page-level signals contain useful predictive information about the proxy target.

Logistic Regression is appropriate as a first model because the target is binary, the data is structured, and the model is relatively simple and interpretable. This also provides a useful test of whether a more complex modeling approach is justified rather than assuming that complexity will improve the decision.

The model will be compared with the Week-4 rule-based baseline using the same held-out rows and the same evaluation metrics. The result will be treated as decision-support evidence, not proof that refreshing a page causes better performance.

In [44]:
!git clone https://github.com/martindiarua/ML_01.git
%cd ML_01/data/raw

Cloning into 'ML_01'...
remote: Enumerating objects: 187, done.
remote: Counting objects: 100% (187/187), done.
remote: Compressing objects: 100% (142/142), done.
remote: Total 187 (delta 84), reused 99 (delta 29), pack-reused 0 (from 0)
Receiving objects: 100% (187/187), 2.25 MiB | 15.36 MiB/s, done.
Resolving deltas: 100% (84/84), done.
/content/ML_01/data/raw/ML_01/data/raw/ML_01/data/raw


In [45]:
import os
import numpy as np
import pandas as pd

from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

# Load the raw dataset
df = pd.read_csv("../../data/raw/content_refresh_anonymized.csv")

print("Rows:", len(df))
print("Columns:", len(df.columns))

# Recreate the ML-05 proxy target exactly
df["refresh_priority"] = (
    (df["ctr"] < 0.05) &
    (df["engagement_rate"] < 0.40)
).astype(int)

print("\nTarget distribution:")
print(df["refresh_priority"].value_counts())
print("\nTarget proportion:")
print(df["refresh_priority"].value_counts(normalize=True))

Rows: 30000
Columns: 44

Target distribution:
refresh_priority
0    17038
1    12962
Name: count, dtype: int64

Target proportion:
refresh_priority
0    0.567933
1    0.432067
Name: proportion, dtype: float64


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

I will use a grouped 80/20 train-test split based on `client_id`.

The dataset contains multiple content rows belonging to the same client. A random row-level split could therefore place content from the same client in both training and test data, making the test performance look stronger than performance on a genuinely unseen client.

Grouping by `client_id` keeps each client's rows together. The model is therefore evaluated on clients it did not see during training.

The split is fixed with a random seed for reproducibility. The test set will be held out until model comparison.

This is a grouped validation design rather than a time-aware design because the available dataset does not provide a clear historical prediction timestamp for creating a reliable temporal holdout.

In [46]:
# Features that directly define the proxy target are excluded.
# Identifiers are excluded because they are not meaningful predictive inputs.

excluded_features = {
    "ctr",
    "engagement_rate",
    "refresh_priority",
    "leak_feature",
    "content_id",
    "client_id"
}

# Candidate numerical features from the available dataset.
candidate_features = [
    "search_volume",
    "competition",
    "cpc",
    "word_count",
    "char_count",
    "impressions_90d",
    "clicks_90d",
    "pageviews_90d",
    "sessions_90d",
    "users_90d",
    "engaged_sessions_90d",
    "ai_sessions_90d",
    "scroll_events_90d",
    "days_with_impressions",
    "days_with_sessions",
    "impressions_last_30d",
    "clicks_last_30d",
    "sessions_last_30d",
    "impressions_prev_30d",
    "clicks_prev_30d",
    "sessions_prev_30d",
    "content_age_days",
    "days_since_last_update",
    "trend_pct"
]

features = [
    col for col in candidate_features
    if col in df.columns and col not in excluded_features
]

target = "refresh_priority"
groups = df["client_id"]

X = df[features].copy()
y = df[target].copy()

print("Features used:")
print(features)

print("\nNumber of features:", len(features))

# Grouped 80/20 split
gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(
    gss.split(X, y, groups=groups)
)

X_train = X.iloc[train_idx].copy()
X_test = X.iloc[test_idx].copy()

y_train = y.iloc[train_idx].copy()
y_test = y.iloc[test_idx].copy()

train_clients = groups.iloc[train_idx].unique()
test_clients = groups.iloc[test_idx].unique()

print("\nTrain rows:", len(X_train))
print("Test rows:", len(X_test))

print("\nTrain clients:", len(train_clients))
print("Test clients:", len(test_clients))

print("\nClients shared between train and test:",
      len(set(train_clients).intersection(set(test_clients))))

print("\nTrain target distribution:")
print(y_train.value_counts())

print("\nTest target distribution:")
print(y_test.value_counts())

Features used:
['search_volume', 'competition', 'cpc', 'word_count', 'char_count', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'days_since_last_update', 'trend_pct']

Number of features: 24

Train rows: 23837
Test rows: 6163

Train clients: 25
Test clients: 7

Clients shared between train and test: 0

Train target distribution:
refresh_priority
0    13900
1     9937
Name: count, dtype: int64

Test target distribution:
refresh_priority
0    3138
1    3025
Name: count, dtype: int64


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

I will train a Logistic Regression model using only the training clients.

The preprocessing pipeline will impute missing numerical values and standardize the features before fitting the model. The test set will remain untouched during training.

For comparison, I will apply the Week-4 baseline rule to the same test rows. The baseline uses the original rule:

- +2 points for low CTR
- +1 point for more than 90 days since the last update
- +1 point for search volume above 20

The Week-4 baseline and the Logistic Regression model will therefore be evaluated on exactly the same held-out rows.

Because `refresh_priority` is a binary proxy target, F1 will be used as the primary comparison metric, with precision and recall reported alongside it. The model will only be considered an improvement if its measured performance provides a meaningful advantage over the simpler baseline.

In [47]:
# Logistic Regression pipeline
logistic_model = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        (
            "model",
            LogisticRegression(
                max_iter=1000,
                random_state=42
            )
        )
    ]
)

# Train only on the training set
logistic_model.fit(X_train, y_train)

# Predict the held-out test set
y_pred_model = logistic_model.predict(X_test)

# -----------------------------
# Week-4 baseline on same test rows
# -----------------------------

baseline_test = df.iloc[test_idx][
    [
        "ctr",
        "days_since_last_update",
        "search_volume"
    ]
].copy()

baseline_test["ctr"] = baseline_test["ctr"].fillna(0)
baseline_test["search_volume"] = baseline_test["search_volume"].fillna(0)

baseline_test["score"] = 0

baseline_test.loc[
    baseline_test["ctr"] <= 0.07,
    "score"
] += 2

baseline_test.loc[
    baseline_test["days_since_last_update"] > 90,
    "score"
] += 1

baseline_test.loc[
    baseline_test["search_volume"] > 20,
    "score"
] += 1

# Week-4 baseline treats score >= 1 as an actionable prediction.
# This gives the rule a binary output for comparison with refresh_priority.
y_pred_baseline = (
    baseline_test["score"] >= 1
).astype(int)

# -----------------------------
# Evaluation helper
# -----------------------------

def evaluate_model(name, y_true, y_pred):
    return {
        "approach": name,
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(
            y_true, y_pred, zero_division=0
        ),
        "recall": recall_score(
            y_true, y_pred, zero_division=0
        ),
        "f1": f1_score(
            y_true, y_pred, zero_division=0
        )
    }

comparison = pd.DataFrame([
    evaluate_model(
        "Week-4 baseline",
        y_test,
        y_pred_baseline
    ),
    evaluate_model(
        "Logistic Regression",
        y_test,
        y_pred_model
    )
])

comparison

,approach,accuracy,precision,recall,f1
0,Week-4 baseline,0.758884,0.670583,1.000000,0.802813
1,Logistic Regression,0.917897,0.865815,0.985455,0.921769


In [48]:
print("Confusion matrix — Week-4 baseline:")
print(confusion_matrix(y_test, y_pred_baseline))

print("\nConfusion matrix — Logistic Regression:")
print(confusion_matrix(y_test, y_pred_model))

print("\nPrimary comparison: F1")
print(comparison[["approach", "f1"]])

Confusion matrix — Week-4 baseline:
[[1652 1486]
 [   0 3025]]

Confusion matrix — Logistic Regression:
[[2676  462]
 [  44 2981]]

Primary comparison: F1
              approach        f1
0      Week-4 baseline  0.802813
1  Logistic Regression  0.921769


In [49]:
from sklearn.metrics import confusion_matrix

# Confusion matrix for the Logistic Regression model
cm = confusion_matrix(y_test, y_pred_model)

tn, fp, fn, tp = cm.ravel()

print("Logistic Regression confusion matrix:")
print(cm)

print("\nError counts:")
print("True negatives:", tn)
print("False positives:", fp)
print("False negatives:", fn)
print("True positives:", tp)

print("\nTotal errors:", fp + fn)

Logistic Regression confusion matrix:
[[2676  462]
 [  44 2981]]

Error counts:
True negatives: 2676
False positives: 462
False negatives: 44
True positives: 2981

Total errors: 506


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

The Logistic Regression model made 506 errors on the test set: 462 false positives and 44 false negatives. Compared with the Week-4 baseline, this is a substantial reduction in both error types. The baseline produced 1,486 false positives and no false negatives, meaning it identified every positive case but also marked many negative cases for action.

The Logistic Regression model appears to provide a better balance between identifying positive cases and avoiding unnecessary positive predictions. Its measured recall was 0.985 and its precision was 0.866, compared with 1.000 recall and 0.671 precision for the baseline. The primary comparison metric, F1, increased from 0.803 for the baseline to 0.922 for Logistic Regression.

The model uses 24 features, excluding `ctr` and `engagement_rate` because both were used directly to engineer the `refresh_priority` target. Keeping them as predictors would allow the model to reproduce the label construction rather than learn an independent signal. The remaining features include search demand, competition, content characteristics, traffic measures, freshness, and trend information.

Overall, the measured results suggest that Logistic Regression provides a stronger decision-support model than the Week-4 rule on the same test data. The improvement is mainly reflected in fewer false positives while retaining almost all positive cases. However, this result should be interpreted as predictive performance on this dataset and split, not as evidence that the model's recommendations will cause better content performance.

In [50]:
# Build an error-analysis table from the held-out test set
error_analysis = df.iloc[test_idx][
    [
        "content_id",
        "client_id",
        "refresh_priority"
    ]
].copy()

error_analysis["baseline_prediction"] = y_pred_baseline
error_analysis["model_prediction"] = y_pred_model

error_analysis["model_error"] = np.where(
    error_analysis["model_prediction"] !=
    error_analysis["refresh_priority"],
    "ERROR",
    "CORRECT"
)

error_analysis["error_type"] = np.select(
    [
        (error_analysis["model_prediction"] == 1) &
        (error_analysis["refresh_priority"] == 0),

        (error_analysis["model_prediction"] == 0) &
        (error_analysis["refresh_priority"] == 1)
    ],
    [
        "FALSE_POSITIVE",
        "FALSE_NEGATIVE"
    ],
    default="CORRECT"
)

print("Error counts:")
print(error_analysis["error_type"].value_counts())

print("\nFalse positives:")
display(
    error_analysis[
        error_analysis["error_type"] == "FALSE_POSITIVE"
    ].head(10)
)

print("\nFalse negatives:")
display(
    error_analysis[
        error_analysis["error_type"] == "FALSE_NEGATIVE"
    ].head(10)
)

Error counts:
error_type
CORRECT           5657
FALSE_POSITIVE     462
FALSE_NEGATIVE      44
Name: count, dtype: int64

False positives:


,content_id,client_id,refresh_priority,baseline_prediction,model_prediction,model_error,error_type
19,content_af865035b328,client_f369cb89fc,0,1,1,ERROR,FALSE_POSITIVE
23,content_2da6ae9d0882,client_e629fa6598,0,0,1,ERROR,FALSE_POSITIVE
60,content_b9104a222d01,client_f369cb89fc,0,1,1,ERROR,FALSE_POSITIVE
81,content_16788821b64a,client_e629fa6598,0,0,1,ERROR,FALSE_POSITIVE
127,content_02141810795a,client_4e07408562,0,1,1,ERROR,FALSE_POSITIVE
128,content_d660fc1fba2c,client_8527a891e2,0,1,1,ERROR,FALSE_POSITIVE
129,content_b4170c25efd2,client_4e07408562,0,1,1,ERROR,FALSE_POSITIVE
253,content_3af54e674244,client_f369cb89fc,0,0,1,ERROR,FALSE_POSITIVE
294,content_899bf6f73251,client_4e07408562,0,1,1,ERROR,FALSE_POSITIVE
376,content_0727aa524195,client_e629fa6598,0,0,1,ERROR,FALSE_POSITIVE



False negatives:


,content_id,client_id,refresh_priority,baseline_prediction,model_prediction,model_error,error_type
215,content_b5716f2631dc,client_f369cb89fc,1,1,0,ERROR,FALSE_NEGATIVE
320,content_8ed8dc20f1a3,client_f369cb89fc,1,1,0,ERROR,FALSE_NEGATIVE
1653,content_461eb9f35e52,client_4e07408562,1,1,0,ERROR,FALSE_NEGATIVE
1962,content_45a8b74d4be2,client_4e07408562,1,1,0,ERROR,FALSE_NEGATIVE
2552,content_35972508aa52,client_f369cb89fc,1,1,0,ERROR,FALSE_NEGATIVE
2772,content_739b312ae914,client_4e07408562,1,1,0,ERROR,FALSE_NEGATIVE
5573,content_cdf12b9d21e0,client_4e07408562,1,1,0,ERROR,FALSE_NEGATIVE
5762,content_38a1fdd6b0f6,client_4e07408562,1,1,0,ERROR,FALSE_NEGATIVE
8139,content_7fa63804b8f1,client_4e07408562,1,1,0,ERROR,FALSE_NEGATIVE
8411,content_637107baa450,client_f369cb89fc,1,1,0,ERROR,FALSE_NEGATIVE


In [51]:
# Extract Logistic Regression coefficients
model_step = logistic_model.named_steps["model"]

coefficients = pd.DataFrame({
    "feature": features,
    "coefficient": model_step.coef_[0]
})

coefficients["abs_coefficient"] = (
    coefficients["coefficient"].abs()
)

coefficients = coefficients.sort_values(
    "abs_coefficient",
    ascending=False
)

print("Most influential Logistic Regression coefficients:")
display(
    coefficients[
        ["feature", "coefficient"]
    ].head(10)
)

Most influential Logistic Regression coefficients:


,feature,coefficient
6,clicks_90d,-31.339781
19,clicks_prev_30d,-18.920491
16,clicks_last_30d,-18.085100
10,engaged_sessions_90d,-17.109452
7,pageviews_90d,-2.520317
18,impressions_prev_30d,2.208093
5,impressions_90d,1.770295
8,sessions_90d,1.563751
14,days_with_sessions,-1.203855
9,users_90d,0.875011


In [52]:
baseline_f1 = comparison.loc[
    comparison["approach"] == "Week-4 baseline",
    "f1"
].iloc[0]

model_f1 = comparison.loc[
    comparison["approach"] == "Logistic Regression",
    "f1"
].iloc[0]

f1_difference = model_f1 - baseline_f1

print(f"Baseline F1: {baseline_f1:.3f}")
print(f"Logistic Regression F1: {model_f1:.3f}")
print(f"F1 difference: {f1_difference:+.3f}")

if f1_difference > 0:
    print(
        "\nThe Logistic Regression model measured higher F1 "
        "than the Week-4 baseline on the held-out test set."
    )
elif f1_difference < 0:
    print(
        "\nThe Logistic Regression model measured lower F1 "
        "than the Week-4 baseline on the held-out test set."
    )
else:
    print(
        "\nThe Logistic Regression model and Week-4 baseline "
        "measured the same F1 on the held-out test set."
    )

Baseline F1: 0.803
Logistic Regression F1: 0.922
F1 difference: +0.119

The Logistic Regression model measured higher F1 than the Week-4 baseline on the held-out test set.


## Self-check

Before you submit, confirm each line honestly:

- [✔️] Every section above is filled — markdown thinking AND the code that backs it
- [✔️] The notebook runs top to bottom with no errors (Runtime → Run all)
- [✔️] No client names, URLs, or private queries anywhere
- [✔️] My claims use careful words: observed, measured, directional, decision-support
- [✔️] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.